# 📋 LOG05 | 미션 05 | Student Workbook

---

## 이번 미션에서 하는 일

**가상 보급로 투입 시나리오 기반 무인 장비 상태 점검 및 승인 실습**

1. 실습 목적 및 개요 역할 및 배경: 학습자는 무인 보급장비 투입 전 상태·책임·중지조건 체크리스트 교육용 실습의 참여자로서 임무를 수행합니다. 가상 보급로 투입 시나리오에서 무인 장비의 점검 결과를 보고 투입 승인·보류·중지를 판단하는 분기점과 책임 직책을 확정하고 실습합니다. 핵심 과제: 앞서 수립한 기준을 바탕으로 가상 시나리오 상의 장비 점검 결과를 종합 검토하여 최종 의사결정을 수행합니다. 2. 주의 및 검증 지침 데이터 해석의 엄정성: 자료나 AI 설명을 철저히 검증하지 않으면 미확인 기록이나 모의 회신을 실제 현장 확인이나 승인으로 오인할 수 있으므로 주의해야 합니다. 수행 방식: 본인 판단, AI 초안 및 역할 대화, 근거 대조 및 보완, 최종 기록의 전 과정을 원본 자료 범위 내에서 엄정하게 수행합니다. 3. 산출물 및 소요 시간 안내 결과 파일 목록: sop/log05_approval_thresholds_sop.md LOG05_M05_forms.md forms/LOG05_M05_forms.md 시간 안내: 원본 예상 시간은 3시간이며, CMS 편성은 원본 durationHours 를 반영하여 3시간으로 구성되어 있습니다 (실제 소요 시간은 실습 완료 후 확인).

**학습 목표:** 가상 자료를 읽고 자신의 최초 판단을 적은 다음, AI 초안을 원자료와 대조하여 문서와 검증 근거를 완성합니다.

실제 장비 제어·비행·파괴·무력화 방법을 설계하거나 실행하는 훈련은 아닙니다.

**준비물:** 이 Notebook, 미션 5의 가상 자료 및 빈 작성 양식, 기관이 허용한 AI 환경(가능한 경우).

**학습 환경:** Colab은 인터넷이 필요합니다.

폐쇄망에서는 사전 설치된 Jupyter 또는 별도 Markdown 양식으로 진행할 수 있으며 외부 AI를 사용할 수 없으면 미실시로 기록하세요.

**학습 순서:** 자료 먼저 읽기 → 본인 생각 기록 → AI 질문 → 원문 근거 대조 → 수정 → 저장·제출.

**구분:** 파일 저장 / 실습 수행 / 평가 기준 충족 / 실제 교관 승인은 각각 다른 상태입니다.

## 📦 실습자료 자동 준비 (모든 미션 공통)

- 아래 **「LOG05 실습자료 자동 준비」** 코드셀을 **먼저 실행**하세요.
- 실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 `LOG05_실습자료` 폴더에 자동으로 풀립니다.
- 화면에서 **「압축 해제 완료」** 메시지와 실습자료 폴더 경로를 확인한 다음, 아래 미션 실습을 진행하세요.

> **주의:** 자료 준비만으로 실습 결과나 교관 승인이 완료되는 것은 아닙니다. 아래 코드셀의 소스코드는 기본 숨김으로 설정되어 있습니다.

In [ ]:
# @title 📦 LOG05 실습자료 자동 준비 (ZIP 업로드 불필요) { display-mode: "form" }
# 학습자 화면에는 결과와 업로드 인터페이스만 표시하도록 구성합니다.
from pathlib import Path
import os, zipfile, stat
from IPython.display import display, Markdown

LOG05_PROJECT_DIR = Path.cwd() / "LOG05_실습자료"
LOG05_PROJECT_READY = False

def _log05_extract_zip(zip_path):
    global LOG05_PROJECT_READY
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError("유효한 ZIP 압축파일이 아닙니다. 실습자료 ZIP을 다시 선택해 주세요.")
    dst = LOG05_PROJECT_DIR.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(info.file_size for info in entries) > 1024**3:
            raise ValueError("압축파일의 항목 수 또는 해제 후 용량이 허용 범위를 초과합니다.")
        for info in entries:
            item = Path(info.filename.replace('\\', '/'))
            if item.is_absolute() or '..' in item.parts or not info.filename.strip():
                raise ValueError("안전하지 않은 ZIP 내부 경로가 포함되어 있습니다: " + info.filename)
            target = (dst / item).resolve()
            if not target.is_relative_to(dst):
                raise ValueError("실습 폴더 밖으로 벗어나는 파일 경로입니다.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("심볼릭 링크가 포함된 ZIP은 허용하지 않습니다.")
        for info in entries:
            target = dst / Path(info.filename.replace('\\', '/'))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as out:
                    import shutil
                    shutil.copyfileobj(src, out)
    files = [p for p in dst.rglob('*') if p.is_file()]
    if not files:
        raise ValueError("ZIP에 압축 해제할 실습자료 파일이 없습니다.")
    LOG05_PROJECT_READY = True
    print("✅ LOG05 실습자료 압축 해제 완료:", dst)
    print("📄 확인된 파일 수:", len(files))
    print("ℹ️ 실습자료 폴더:", dst)
    print("※ 업로드·압축 해제는 학습 수행 및 평가 완료를 뜻하지 않습니다.")
    return dst

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 5개: LOG05_Colab_실행안내.md, LOG05_M05_forms.md, LOG05_M05_lesson.md, LOG05_공통_AI역할실습안내.md, LOG05_공통_작성안내.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAgIAAAAIVzAhdWSSQcAAMMOAAAbAAAATE9HMDVfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kbVddbxNXEH3Pr7hSxSMmDtDAI2orQCJq1RaQ+hIZSEtK'
    'qFECRfShWiebaBMbYkOcbJK12RTnq3XbTbyOHcm85Kf0ce/d/9Azc+/aC1gCFtl3556ZOefM+DNx6+vro5eF2i7FhaY8KQhVbMTry0JVHTkfjoxkM/rE5PP8'
    '7KN7+fyjzPSTF7/ck7s98UV+JndPbZSV7Qm1EUYtX6iGJU9toRYtfG5eREC10o7XbLnjKb8Wr3Qzv00/UTUcc5ty770sOrLYyAjl2+o4VL6n6hXZ6oofbn4T'
    'tXoiagOWHVcP++f6dwhVb2q4qmaJqBuonRCfVZR9JOLVUL5cI5jy5T9qPVRvArWPN6orAGNCjYxlRNRqx0ttce0mUvg7rvomcbVpiVtTc3P5Xxho7T1lB1Sq'
    '1hVqvgmsIi6VZfHgrKNqrmx2zzrXblLOcn8n6uJMrUcvyoOe8nGrXxjcijC2XLX5yuB9Uj8Uycdja00d/33WkcW2bFl89bserpbBdirAtW9uini+IVfWRFy1'
    '8Q6wL6W+j9rlKLQEgiDSv11Zd4U6DeUbD2DNPfVd1FDOu0KeWLJECKuyFVJEnYaqL0p/N666nxbtYkZ8+9V3t299L5C2qpfRepRAqPWKKtYY8hEwBVSwudyv'
    'U5OzU3PPZp7ykXc9ZfORuEqNxr+Gbun2bnaBScSvumq/gp6CNUKubqKggrhx4ono+D1RCQEJgO1Rk3XXEZAqK8v9aHEV51utpMrmlMtc1aWIOr3oKOBCxZtV'
    '9Bf/NX1ue+qPRaFsP15g1HL+IDotJHW4lNH55aYnH0znZvI/PZsirEM7CX2VCSyCREEBf0CCEGxN5x06oB5dDvLZeP55QCnGWyVV9NG4fScJ8LEg7j58ceFG'
    '/vmFuw9zXOaoZcdbL8860bEVL/n0RDN61Nsf87OP53R7dSH6MVA2ioszYEO8CUbt28qzSQaa59w/t0WFizpduWdrXXTVXm0oSy5nOGfoWLmOKnK7SKENi0XN'
    'RRfxawd5kXihiSjwiHUG0LdZ+vh61hxVG03qrKqvybfLV8bORYFlAl8dPWcCM6KTQ7VzNAyRkEuv+aY3R1F3GQX1y8pxSbamxvz6fFPWS5yraf7KKSXp9bi1'
    'QzxEqKUSK7Belu9KDAtaxPlDcHsTomqj45Zc2VW1kPSmGhWOsWnjODGXSOdXh9bw84yYGIeMG2orxP0QMLkeIdHOOjE6PqkcD8yO3apWuHHbzMOnj2c0ZUPp'
    'N8g+qlS9MucJ4XRLEg61FcCd+g7ed+LYteGsxjPkfkl7QnRcIo+WxWaatXyGXBTtg25jOxgDOwVnGxrH5Hoe/9V3TN0uPLm1Wr7abkLxn+UOjgqCV9/9z9rk'
    'XBxb7vUJghvibTtlppq4JhujHvkn4p6edQYdNqWQFYjglPX0gSlQsWulJHUg05kNMh4Z56YMMBKnNEzBeD6eNkSFQGhV01kcUksvBTtbjSdG2lV08ULHCBZJ'
    'Ks9nVtUdxGUNGFkVvSiwh1CfDHLBMDaueDg3XA5UAR4D4vaX57Nnndt3zo9xpxYK8QLnDixyr5lUpmThG+iGJYgn3KGOfBPb5BEMZygxQWVlp3+RaUWrTeOO'
    'gKZnEN/ouNrj/Cpw0uu6OkNxG8aR7FYaqB97VnrIDarFn+81aXx/2A2qyYbLTak6ifeRwDHZtw6HavEKEoFVAd7+coKB+n3j+4lbnN6Oh+w/FKs8tuOVnvEV'
    'Djzw274uyZL4Xar4k9n8z1P32cVplhVPKQW1iNT2amyYvNWwgzPlAJWNxpgVz0f6Nl5F2w5Sc/BxbvbRZO7+02e5mcmn04+nmLW0QZltj8hUIKn3Ky//pU4w'
    'H40YMU8Ag96kpA3qC6ab8YJFmj/o0sgwAxSRMbkTqg8pw8hViOkKsh0bhUIFDXzMroqXiPjESfYbdk0KTY8NdHAJVtYQRCDvIDVWCeN8kDaobRz0EH+dODvf'
    'jI5Cika+TDPlBDakFhwQxo5oBL1mPmp6DOYmNcH3YAWUIqmZGcNEw2ZgD583E+OIdVm5BSyWvBNh4NTSm4BZDiEuPQB4KYErn7omJy2sVDLkcqwmroQp/Ooy'
    'gMDax2B2w/aDkewowFylPO5kf78zzhkaD4G9+DbaCwA0lpPFp97Uvdb7Yndn6Nqhly+QDpPBGhvFl/31UsIxK95F/RENePci6j40JX2UANGaS7PmksjGr1yC'
    'pr1Ab4LM9MD8SEiW/GTntAZv0J6oSZPM4pTlpRF/4kTJ533nxBgcagXZrN4ric0YQoWQ28pbopi4dNaZ+Bx/r3C2LEMoOTuKIUK9Y9d13I9WXs4GxedtRBOE'
    'CldQfkUkW5zxHK06rIe6bgNtJQulNuuBVS0mA/7Zk5l87sHcBd4d+j/BDH/TayhiQliOqS7xoo+KZzDclBIODmAKukDUN/rRtHXIU6jYgECAWJNME9SMJwx9'
    '/ePELFZ69cPx1Oon3wbkJnXbgOF1FH3nWZnY5qAd/wNQSwMEFAAAAAgAAAAhXIebYhfbCQAAjhwAABIAAABMT0cwNV9NMDVfZm9ybXMubWTtWN9PW8kVfvdf'
    'MVK0EqgEhx+usrzBSk0jLWqlrbQPVcVGyrZ5SEoUVq0qociQS+Rgr2K0GAyxkZ01AW+NcjHGNqrzwp+y+3Zn7v/Q75wzc32N401UdbMvUSDY986cmTnnO9/5'
    'zlxTn//h1o2UWsSvOdgy3qkyO1smW04kzHnJvNxQZt037ZJu9ObUVyvLD5P3l/92I7V05+HDR8v/uHN/6Zt7j75eubd8/+7KEt5OPrj7VSJx3ZoyB/nLjvbr'
    'MPdTettUfX3mqaQK/JIpd38zf/undGGuP7jcw+CmZyoeP8zWTKWUnL+t9A/HplxU4fNngZ+GSbOXVkHXN7W0csMT979eWVn+Oyzl9fc5fdhTpvwmOKtY02Gh'
    'rrMZna1NKvMir8+6WOisZfY8FW6VyHbg75kyzK23zH6dPoVPzzEpLBTNbkOFhRJWyuijXGSvpZ97Sucjs/p1F7sNzs5hGR+7VVMpXHZoXPZCmd2nppwzZQ+7'
    'aplShQ4QFo6D/xSx4nFwsWatJBLXrl3Do+JkFAqvBN+pKQV/lItmvRE0fVhW4WZXv2qMwaPBaSs46ynd9uASPvipZ/axbG08kVhVOnscwOerKtwrwOfKHGV0'
    'o4vvppiBw5Uu5OhLwcMWTLEWPcLU69evq7f8jzdT+DyLIMLImtk9UaaSD5ppnI1CpbfyOFw9aNZtkPjc7bqpwm1FvbmNIGI+/8DWNC3P8xXORkcx2Yp+6cO/'
    'NPDW1GXn1jR+Z5SpbSmysF7ESogtEAVohTuHWHXDZEsAkwrO88Y/toelDZmDzNCSM/gQNN/oagnjG4HvEQJg3RyllXiUt1z1gX6TKdJkeB44s3bDnWGbs3SM'
    'zQu8Ddo5oHBMF/MYhKMVKICm5JkD/OTHeS2vRFsesb0UmTrwdKPFR8bS+iDHg3mLYQGozhTtVPrIMe4RZOE6GohtAt46D/BeNZ8YibFpYExQhHCoPrLM6QZ2'
    'A+Cs4ZPSR7WwsIWsYXBFw1fthDFsx1wA1nL6oJU2vo/sSuvNQ86iRtccHI7T+LjZMUrsJ2v0nFBcLdMhaaxXegcUF1N2FwzKmBf7L6ZHvZgZ9WJ21ItU/MVo'
    'V86QK4W0BBP2VIjemI3s4vzjxc+YyJrgKc94FbUw9XghJUmbI29Q0ubyCO6YGGHO0oewCQY5Sl92TJtD6zJ7f9s0T9iFnV5w6l/1scQI9EfZia2NgQFgw2X+'
    '+M85uu/u+ZgHrHcWhh99NvRoYWr40fTwo5nhR7PDj1JXHo2OxCwiEXRbQfcZ0Yp5+m0UkE7JXOzNDT41u3lKTTC7qXlE1UQIhU3JY6G4tBpTSo1PUFqGLzxA'
    'O49PcOmJ/qE+8jXCD/RHr2FaapHlHE7hwcSVlWmGAIb3MHACAYlsWcxy/QAOWqBlZeubLa0X3URCaoZKSUnARCTzgWn7DiZRalMVSWEKgNYjkiSAHRzGaqgt'
    'KtPYVJ+fdblHBVe4ODZ4cf7mzU8U759LL1eebEl9euMTxZt43dXZBhF73XfMKOuSm7AVnD7cPw73tqkaW79snkfmF9h0jHqtEaUzni6/mQAaaURYBP0eckl6'
    'sgbcw3Xt+mXnSsGgShVLqtgxboLdXHnTz/eoUA5oEOcrzljslMVHbFdYMl4K8PUtwcfoWPBhsd5y4UYwSdmQW77zh9xgE4AhMcAg5vsetkSSAQ/PtsKdZ5Yo'
    '6B1zNXbC/sLft+2MxnEFQrBOQMuslliEYCAC2ujyicD578MfQ7n8qzygNIGupKzc/xYYaKbDp5VIEgpX90Us8q6E9J9TP+5UaZrOnuuzNCUYlQSSBfTCPjW7'
    'LfIV5b7Mhpn528B50Mwhw6pBF/7a3QCmifxFDrLaZfFo9Su2FIcWO1q0k9VINCDjtEB7G8GYo6gPqhSUW3qQrWF+uWhVH9t3FcJpFIY8o7cA7bjNM2XglZgm'
    'rrh06K9AD7mqiA4rjEDW8Q51Ahko41yOUshBx41yXhwxCvTBaQqBUd9GMr5DHIwEwZVvCdKDhAE6OUWY2wAlaVT12R8v8lFsQJ7rPpyarVE6CVyiYs2SicNK'
    'p3KVGplGCWMLOAz7RSLpXzJzEjgIsp9Ib4a4izYF0gPFubP2+VyAL86n8mfbIYEJAQPDiLMdspirUT0IPf2eSU5rT8+tSGUtTlGrUVFiLAzxh3tNydMPweDK'
    'pZi6se2pVwmflAnjtCy1m+xaC53/FSBEEZJrItLIYwr+C5+w8rWlTFpSW/2jrI3Q7JIV6hFgxXu39T/Of/FF8nfztz9PUrPIJ0xamtbtjOh1cYZjJ7v2/wnw'
    'P3tyJiN0/NaF8Z6cglHOcbO7UbQVT3tFIrXorK+7TpBCAhVOpL/PS5+As8wxITp5LIfrt/vUwKFpQOa7Zt8NEZJ13pJvcTTxE6mVdNUg1uxOxO/oD1kouosE'
    '4l8uqraQ88ltXMGDaLHknJPE8abZYjYtt8K1ltWJ0n2BA/qpBBN/lsl/6XdHwGZcYcFHA72B+jHtBEzSKXxupvoWxv7kLlnGKYGnbkA7OaElSoMXYPET79NY'
    '0PAZo551ItZxUGGvpqH1Xm4QbUkT5nrPmb7gIZnWZzfLCoc916vFTENNVTzWFXs4NQpUCdvz0JkiSj4vCR692GOStH2t8wPvFTPWa3Sgt4l2Oyuuq6Hioaow'
    'U7AQbYUC8eW9fyV/v/zP5Jf37nwzZ0uHot1QzXf1hotVlJnC93G+dKoAM+J0Q0telYgxENtciHAul0KMXCpqsURnN543EGnLqpIAEePQzZHTgDqPELB3RklA'
    '7Antyx7dspkjeOMZ04d5VY5EzaodoqIboSHSYleIGpTSzMwTyaN3Xw3RVaKsMsXYdtVEboOENW6KixG4456pUNF4g8jhHHNqcfrx4qwdDTdyp3zZ4e6Y0fI+'
    'qjUenxLXOEdKr8oEXu6mRJfEhZduZsIiN9oMFlFwSbkz43CG+zlAEgMYKEmp63w5aW/62GnmYMOVVsFb7OZwh56zrhf50Nf1dBXJCT3c5NiGS3BFkPnr8qMH'
    'K0m+uV2Cu5f4++SDu0xqAlRbAN/nxnZCDVuaGL0G2MDs+thMTEKs0j97ndeP/zTHH/oX27J9p726dJxo2004SH9X4fBLmUBHGQFRyBC4Fy1luxF7J3Il7EWV'
    '4otJDlns+spewBDH2K1/hMUHhMXMACxmpOQNtpUtLl2VraiFLjEcLFEU+50sxSHWyHJMK2uAB9Gt3NMZr8YXrF5fwcuFS+bfVCPlygHc2M7xpSRfFLCdfXSE'
    'OWm1yN1EnqRiPmLlA2JldgArs1JCrByoZIx/PKTIpddI8yWVoCYFIFDOX3YIPxFpRBcXAyqC5G4hg/gv/vays/gpfqdu8A4rHt9MfYz+B4v+fwFQSwMEFAAA'
    'AAgAAAAhXFeKwKXUCgAANCAAABMAAABMT0cwNV9NMDVfbGVzc29uLm1k7VnfaxvZFX7XX3EhBGTwL0lWk/jNTsENVPRl231YlmwgbVNIcEhCS8EE2ZkYxVIa'
    'ObHikXdGHe0qsZUq7NiWLRkcCv5Tum+6d/6HfufcO6ORJWXdpSl9CKw30sy95557zne+79yrS+LXv1mazYoc/v6V3xI9P6+erAp52O51n8m6I4JCS9WeClV0'
    '5Jot37aU3RC9ri99W8hWW7ldoWpv5IklMC144gjllXsHeSH9F0JtnPD7YkNtHCUSly4J+UNXPbV5aLWZSMjiiSp6ys2HyyrPuaIKtlqvq8amLOXNEsYb31au'
    'zYvXXSGPLWW5Qu0/VTUsvv0+qHhwVBYLstiYvtA+1HZZWc7gNmgFs4Xegd87PJVvTtnKoReZ4G2ddfBUvrHPOnBV7eZpXFAqy+JeULHlxhY5CFfJ1uFp5Obu'
    'Kn1yLRFUK8qrYCgZ1hEKKg65nkhMCfVtWR5iCfWkIJb+9EgEf+sqa19k4H3Pt37Mb9GMbr3nv1RuW6iKJYv2j/kKzTxy1Hfwcc1Xx45snc6Lbx4u35+5u/zH'
    '2ezNW/fvP1j+8627Nx/defD7h3eW795+eBNvp+/d/gZTe4dHwfoR4rBJa6lKQa61VTUvFm6Y8JpnHJG9U+XlhXI/wHETc86w9pwSCXOwVZbfl8hIaNVyVNHF'
    'BoJNfHCwZaF22srmTKmqJSjJ9irtBdYuURqnB42JlJgazC5eWMihyPT220PpE1/1jsrKqaudpp6krXydSKyE81ZM6s86vaMWwosH0vcDywdK6PPuarDaEsHL'
    'gtot0GAPNugFZRdOrNCm5LunYgUmp6amxIX+j7G5BXz57S+nUpPiuoiWvnr1cnIpPYFPc/LVPraXXMrQtytAlnKsJMKudrqmCuhF8PK5Wn8u9GCRNpN1ELBZ'
    'gIjx2jsu4Vvc19wirf+7qfSkWOyvfyUbrp/prz+JTJcoS4ih3HhDL9Pj/EFagezQn4xxP/LHeCB3S9qJ66OCcC0FJ1I0Lxs6wS5dvVgQMkLZBYAjuTR3oVgk'
    'xoItTWA7+NA7BNkA/2stWc+LLFXYR3AlW8cIF0EHBbTTJA8qntxtMNrdtnK80VjBs8WUCUcMglTIPmbWtuTfn1G9B5Vm76AJhwyBpS4L+W6PHKjRqtFr0euc'
    'qgNbG06bZJ/DsxnstsGKEXfXnV6ni22JVLBTkv8shzksevI7X76wyHBv38e2PgSVPW0/0y8NIjIMIn6rVoJaIUwbUBTPG1hlHXFKwnE8IOvgQk6mnq+9J9ea'
    'frQiT9ErzvXzrtNNPBPYFqpZgByxfMiPyDMUABwl31r9wsUATCIxUsdNbTIboQX8ZPBB8qRqBeWccohiAKIIyWMAqxVU2gJyRbLidnlJHkb0fuBhXVYPipRA'
    'GEgI5MEWPmF2ATIxAEIsMj3Ilsx4IDagD5xGtRVsdLFUUtV9UB6RnNYaIma1b4FQseQEI7G4B+hSnCA3pHG7Bdnqkht2QbmYVykxOqEfBRLE8NF4JsMbguhc'
    'jxBpr0IYwvLCroFCuVk2mOIwN/Ikdoivqu9rXaSgrQj+D7bSUcBDvo5ARgOXUmedpTT+UNGNTUEW1ggRiCc0RPrN4PUbQTkqOpAMQSXp75nNchpqhaElCago'
    'aUKE5hvWDNZwoSPKLtd9VDl6EZqMyCvPMnaD18M2CYpxZCSlXaYeAMmiBOqk4/kEr2VgNdo9hqAWJtoyUU6txIPZRZbhgm2m0kfOMXqP9SPupdrkJrUmZVtt'
    'nzc/HmNEdFHHIvrIGmhcBJPYJmqDwRUNXzETktS7naAx1LvvtfPK941mEPkBeyhOppK42aRm0AldmmAG2iSNBbd9HIq5rPGCQRmLYv9FetyLzLgXc+NeZOMv'
    'xocyQ6EEH7shW5hdIXtJk9ncwuPcdaKrvrYsph4vZnXRlvK6r9DtZHKw3wSYPFiClB1zasPK3tlSB+85hJopz8VY5wjdGVUnXEuCAWAjrPyJn+5ewo7lXHQW'
    'hx9dH3q0mBp+lB5+lBl+NDf8KHvu0fhMzJF6d3WPiNogoTAJ6TjqpDo/+FSrKStXwyKdJkKobOg61hSXF0khBLVDa63gW0iUX8YnhPS9fNcc+xrpB/qj16Ro'
    'dEQJOYdLeLBw9cpaeQgw7MPADjRITAPAZqdNGNh0qGHc5mp+1U72teBcwgcKOgt6Z/z2+fAilU0aWXe4DdimIwFss2uD1uUuvxtJAUQ54QGPCYA1bduHOBDv'
    'njMZa/x54shGTYsSImXqiUdyjZBtQzijbOtTba2sDtpmar9NoqTtlsL9ckqQNw4NSwurxvY6H+4Mg4dsoDs4i9M+uCTJl2dxpKuoTZC9g+OhxYeyHZ/2hZiw'
    'ZnFHMjg9oY9f8PCgEJ2z40crOjUeozV3uPXV5KGsBqSIT1u6p0C/zS0DhUvu1tSxH+Ypon6KWladgHL07glONYCgGR69TdORRjT6+i3dUwqG1urY4NwCDjyC'
    '8W1OjuhMio64NntZsBM/dGWxRcKPJtAop4m6q0+irXawsxdUtzAmrJvoSAp6YtMjun5ZsKT7YRJsRSNM26g7Qzp3U09IB8KBhsISA6Qb28ZVQDBsf+SLKjVS'
    'uv1Bp2dRUk2swgsCBkbMKywZbxXCS4VBctAgCzcJi812dMdStXD64LC88ofCwNAIKlU8gR1QXFVzhf4M7dQ3P1W+idFdm/aXt0X77p/3ES2YTCRy6ce5OTMa'
    'JcWKdtZhFWOsM19gHxxt/Dt6X1FTFd59xNxKs1sLN0gnDSxN5wszDEqDRlSifOWxV5rrADhd4RjIDCYQby2aXIzRFc45b2wmPtMlx7ofo9+Uf01FI7zNsLeh'
    'fX0dRV0b6M3bjBLvsJcmrIjS96d4g9STum8Gr58ZHeelvFV4vXAj7D50rXIywP310Ak4/w+CtS4UQPa4xITM8GY7Oz5Ocbr5ps2SkEGeRmxhTuPACKZXAPR1'
    'j5IP1j1zv8d8jFMcVbzeTBb+UYTOOrStKMT6qMJfY1JI9FPB0cPK/eKsk7uGv9QslTlKhMvc6cP1xTPSBLo6YmwbqPVvxwyJGq4jzbOIqalXd9vBatvIuW6S'
    'Edp+tcLEV3ry131VCjFliA4JGWjhdHb580zYiGloRRaSX4S3ahMU99Ts7OWI73TB8wLMQfF2mpPEe4yOFpOjtYyly9yU9qXvwgoXmv5JmUGA8Y4LYEi8SLHz'
    'wRofaUf1VmZWvP3p0IUQA5eFKHKFEvHlnb/O/Gr5LzNf3rn1aB4DutyewZud5/Ct2ABgCO/gd0Bbd7Ya/nrr1KYRYWvLmGGuPy0veOLG6SWWe2NVnzfPOlRh'
    'nGpYAZTwsuicdXRZ0gGdmhwK41ELmTZChBCiBYoqgy4l2lEbUDYH/rFMrJ3kvdqm9uRh6GqE89Q0X8rr9z+DoufFJ6Ho6MDx1iWAcWPK8Ry4OEMTEth8Zunq'
    'u3Vqy2c0sXLIQUmADQZwMmfIF8II4lFBd3zCOnhixy/0GRN0baiN6ZaNJbDYYOaMJDC8Wx7RDxgZ6V/t/2H5wb2HM/wbyE2E+yZ/n753m4lHgykolUjPL3KN'
    'PimGLU2OX4OIzwhKhM90PO3/uQTOi/+pBH5Gw6dFQyaOhou3GPPi/6fF+AyRTwuRuThEfk4LNy/++y3c56R/2qQnYr8Qf+QH4hnTk/HvINHPxNPcE9TpzoSS'
    '6dpB5UgVW/rXKv0QOTIpQkXzhWNs8X8DUEsDBBQAAAgIAAAAIVwFXnF7IAYAAMkMAAAkAAAATE9HMDVf6rO17Ya1X0FJ7Jet7ZWg7Iuk7Iq17JWI64K0Lm1k'
    'bVZNb9tGEL3rVyyQa+smaYO0vuUUtGiaAuk9ORU9NCjQ9gdQNmXQplpLsGhTDilQiWTJqYLSEmVTqJyDf4qP3OV/6JvZ5YeSnAhR3J2ZN++9mTvi+6eP7z4Q'
    '2WKZ7y3Fo2+FOnmXe5FQ7kgdLIXyHLmTNBp37gh5aSs7vLnK5la+hw9Oe2r+TijfUW7QaNxagfJsOfDNhfLfVLV8oQYd+aat+pbANULtzFRki7zdke5UhbaQ'
    'cS/f9VSwlsNAqMRBOH59aOOIGvlb5gL6O+97Kkxlx5cHPb7JHSGZZUcFQ3V6LrI0ViNLqAh5ODdX6rKXxZY5pMJE5J6NlHMvoExwgOJkyxmqyj0/W0QmkpBz'
    'J/dTOV7jqqY6TmpZqJMO4Nnj+IOxXNlC2RfyYIxobpDFTTwDWw1sPO1Avm3R+1F+vC9U36YicE5O2vJopCbAY+Jsicfy6CJL97PFGvmdZ/Nz3I+ULAqPzOTZ'
    'TCf3GcOnS1ShL6Rjy/Aa9+821SWXouL3wF8nuwzU61aJMkPPWHhRNm9v3VphoyGnaxXRqz66jOrofzVx5CylXIpAuDbyCnhAiCgwbWfw0lgOQ6QtXUe6oy2Q'
    'hwDSYXHQLuqYDOk+lKy8A0SrPjedEF9mF+BGmFA1FPNVx+ShJk0VdQWQRp9NHnKRmo4ylvQiAb4msZsr3I1umMZTu7OrdXYRcyY702zVNPE1p99OCc6C86eJ'
    '8puNxufiyf1ttLiL9gIbBpz7xGUvllkcUDJpTFVSzVncJ5yRHvAq0NOk1l2UnYDq1dLhVOJXNSjkIc6vqy+4GxpvediSZ9ciP9zHS6IhETP0IS+kAFCZjHVc'
    'KfuvtnEX/zOMoRSRJZaKOiYjjQcSs9TwAgrCtaOuWvki/zvNj8esv5WNNtZai4qBplAXLU3viZct8JRzDyJB0mBCRAcJdWbMR+2mtB5QWlSbpo6QbiLd1c0V'
    'nEe5UflTHawQLLtso9yyNcxNFmAUoAWVWnvQHAPtLmsJlzLtGOHdWoSZPnxrAe5EhmvmE51LlMcKLI9zClV8poC7lAuL9WRYE/rZfEY3nccfKGlnJgft2tkP'
    'JGkXzCaL2YDo4bbIfXjITKhVIo8CcNzoSo3+KumixVsp4T5YRgJij53/U4tTiJSlk12lIn+F9r5tyWglCNABmRdI8rqlHL8oLD9tox+gBh8mFAobxdm2lfd7'
    'dag0T/UZo13+ELIcJhv2gBS7QBHFW+okpsSrZAspSTBtN8yP6Yw8TzhndlcbQ6EIls2vYQ/sNodwyGktCppMQ8igR0rDizPA+Wb9SU5+vS1k5ODbvG1xRldT'
    '5V2Tb1BCiPVg05wKafiUXNUB1G24RiFx1TAGqwhQ2aWifBl7QNrxIQ/oZpHIsc+3BlExajbS+mabkMi7ATeMoGMBbFgloVWZ4Sb9jXnoKzR+nH0v5kmbDlmm'
    '57WI9+5uYx52gBXrWs4S5AsLxLyu8RidixgH46rkJ2xZm43WF5QHKzfD+7EetbO0UDrjh93B7det+clDoZwAgzT3vWIbAQUwshuNF7xjPH9y9+Hz8hvd8ef6'
    'y61f/nz56wuueBio/2ZCpm3oXZ3GEA4jcHJdYRWgTnqIn+6R3rTDY9FgzfAP7oJmXYkmVopugB+yU5GP22EYjfIAVbmZQE7lIFIDCHBsVAln2pABXH7MDmXz'
    'mD9rlYmGSaVeNGlTrMQtN/qCPjreZ7/CiGDZCpAl32UxFHbAfkeTMmwbqbAtm8KresBd49V6w2H3W5AU2XJZ4vXvJw6NpMk+cfO7Z09/IPihYBYFop0m+E1a'
    '05X//NvvL/+g3dCwg4jD9qIXrRhzpcbRhu4+0/BDvMqmWAZbs3JlqzZRlo4UXSOzhsNwMZBNM5Q7QVmdzdQ87JcK+XiQGbNNHHCBRwybWU3LtGUAs47PZrhI'
    '9QGznv746Nkzhi7uZe/Tjy8vmqu7ysrQkwzkA3uLMcY7zheF557soY+gDLOz7im8OaLbGKuw8IDXDMtAoy2WN7ujT2xnNbCNaRug+XBBfSSmNxMujPddlrke'
    'MJqSZCN6WR50CdpqJd8QPZriMJnjaSnJiKcaPcwoIk1s8Khkx/9QSwMEFAAACAgAAAAhXA1xTZ/1CgAAohcAABwAAABMT0cwNV/qs7XthrVf7J6R7ISx7JWI'
    '64K0Lm1kfVjLbhtHFt3zKwowsslIsp5+LA1MoDEwwgyyCZDBwEhmlESIIwWSjCCAMWhKLYEm6ZiKSYuUmkzTlizRQ8MtsklRMLXxpyS7rup/mHNvVTeLsj0L'
    'PdjsqrqPc889t66Jv/5tcXpBRN1evNsTqrGn3DOhKjm5FWYyn34q26GqD4TshtHgkWocywtXxLm2auwI5btCbWfjbe/duTrbUQ0Xf4/21ImjmkF0FgrVCeNs'
    'W75sq/xRnB9gs3xZyPZAuVgh+65y63G1gq1kIScLR1MiChxsiI09cVPlqmq3if1k0RHaCm2EDKqqXqWXYLOIBoFs1lUdptQvo64v7txVNefq0e/Oo/NhdBYI'
    'nCqPqzj91Slvsv86rvgCR8S1Mp7igK4PL+DCxSnMlMdDE5K40kqsVIUjslAHg96Iem35pBZXqliH0OVVvpe8K/sONseO/DLM6LWjgOIE7y6qdFKWQlfHusOS'
    '7A7IqKhXUl5THbTSeLg40IpTJnPtmlAnbvzskVA5D3ZmMjMUvay4v7yxsbbKrjVK8kWRHdCB+WZt/YcNJLKnttp00n5JxLUKwhpX4MxWG25NCMRFHoeUJzyJ'
    'K68nKJ77pahTFPKkGQ0G2G1I0ZanQ+XDaj9re1voya5jfS3DI+UNZSsUvztV/S1iHspGUcSHrgnt706NlsR7nip4ozhnZqdEXAll4ULINwO1k7iEoNVDwh5S'
    'r45wRsfFJzw9LCEFZHrUCaLuEE/6ZcQN7+WQGIoE+0K4lQU4V6WoJMk8KKvOayzp/Nf2ohuqZmiZNDdFe1B0XwyRUjHHMIc521mG4GEpsZFzOUBZuWorwL44'
    'Jp+gFcYR5FAlFqrkSYPLYytE5uUTbOb5jCfs7vmcRz9LFscVF9Zy0gAi4IIOBoz8PeOFZe/8FPtQc22nYErBS3PPUdDW4iihztx4u64B6le4JFAwjPJOgCd2'
    'HfQ89XxHxMUiMME+oTKrgKvnAtu0RG6dRhfZxJiFKWEyp/PJWDep0rnEcW8G2rDE0noxedvKXGrC0sw0njNc0/K5FQVcnGwRyk+zA8gN62zreQFqy4e/Ymn+'
    '3fnSDfzc4tD7LgiEi6IOHglxBpEBHZOrouqsCGe++O7nP5zyKN3xweN353fuchQarknfH05lQvxl7Se8aQCXUE7CSwwHDjeypXmBEsqw4dVffPfVJi3XQU/i'
    'Zs6NiyVZOMVHOwNYRkmInzwCj1ie37mLz8BbUT4F/msuEb5mPjwjXlV9jyoJVYszqILwEkBNq1QFGajZPDSCtDZCM1gbRJvJTMIBXz4PEFMK3uIMIYxIDaUT'
    'BRUDvsUb1mNVubRM5d5EO2JjxJaiKV+6RJf4Wh7D8bIrm57y6yB54n1+HMo3SLsnGM8pLpK1Luf1wMYCyGlxztQxs5EDW9+dL87iZ26Cv56nr2EwvmZXgJfF'
    'BfzcIH7DWfLVzjh7TQq5+yvtZDKcRoLbkw9KdVDKrajTStx5mfbBz2c0g5flb4/ErdlPhI6e2UrttxmkBsDVHA7m9/ueZOjzstvTnxiGZJf7LdU8My1Kvnr7'
    'sXaWLgl1qos6iAxDzqzpSPrQUUeaNC6BxC2LEDh8oG2pcv1jYBHVewo8VST17sCR+WMTcG02EGgCqmlAlj7QVNVuEUyX7KkZl8G5v2u6DZifHug9TCuzdQZY'
    'edCMgl+JmD30TfxlyYIKMrjSJNQosdMc9qTT992xLCdi6MUQ+Gc6hiH5C6xOyNMwOfcj2yomawGFI6jatqqWgeluTGLdPe707KdZxpjpulRYBD6bZykToCmc'
    '/vQMconJKGF/h2KB7stwYSOZZ0E+Bf9D+oXy2nChvthM1alCUVlFaGkWRHT38QRSWsJf0FrnkiuQPZ9gUoH5tLiSY92IQNEXgEF0Ti8cAj0tER8UYbrxcSI9'
    '+/wUKecWyPuNV66WnKYuSb8Qq7GlhopA5WidQnpvx5zl18FXz8L3iA306qiXdVnyUsAzPCgbOoBGOZoWlnTh/QAfxtBBbePVDiGhp4JL0p070r8wjUx32YLP'
    'ioZTzDzLfQbrKLFjtKA34YJs0BNbv3D1siMcWZIHUWfb1JJ6GnxAmp5kIZG5FWTz0SA0Wn7U6xJb2Ub2r4GsV3lDNjSVbB8Bjh4WisUopMJqh6hW9DyKJboT'
    '0SCaMraGbgAZh9xJOrp+3CPqpB8S1EYRqt+yBPMUg8gq2wXPvWoUOlpR5BBabEkYIm7a9VlEXMlSEkKS3b6bdjHHPGClXuOZgLaGWTAa28hyMEqNVuTNACVJ'
    'GIECQWn96c7dTCYpEUfM/+cGImreMpTKgNLH6zaNKhGz1EvqWoX4DAPLWL017RdXakTKjRKnkKcpkY5hiTTTvrjCtFrd+k0sGiVEx8/Jp15MI8rYSMSVctK0'
    'Ak8sYt7ABID8ebxxgj/k9BD7jbzQb30Ufck6Q0chkTeBDkQtX+mWCNnHUpX7uj5ayx5WNZXT6G3VqA9WK1pEkWxDfAwPA63PA5CBcfm9TpDkBxHbzonFlU0R'
    '/4Lx9IwWz97k3uBatLx0k3e9CBEzSqKOLxLW19JkK5wQS7dHU27S52en8QaIeF7MxL9USWdQO5ijZejYqCLmNi35El0+Ggz0VnTWE8BH20RmjPVtc5L+mrvG'
    'GHUkYkGjLqAGDx90R9eu1DzMsGY9CxPaw8xdaOgYcMzJFOAhhPW41tLKl3CBnVtBat2gySOYzttHwKCrR1toxhWdqExmaYFBkR9Q71jQKpAFFAOd1Ma8uuAm'
    'tkCyX54cxZW9RJKdNNS5a2EO6TPDwohd0etH3ImOiQ+EJiMgL5IomWG/m9x48HTTd2GVriz0qkIdX5tyNGA2SKH/7NZvJTehM32QsU4LOJYnuj+mimdM6jwL'
    'Tfs0QkqQMOzzoEvn0KzmC26nePl9Bh35hmIp1M3AQkeMximmSz6JOnglZ/NVOqZdFUVi6RbXzzS6OY/EmDaTaxojK3OyPsTwTGoVeoBsONkjBKES9AgSdXuQ'
    'fkkaUJswDS8gJuM0oLbrfDnzgQ5koyrp4shdrZXJPDQ0Ix4mF1567BYP8dXk5KQwv/HJrNT3OOKhdUX1f2+oJugs/Gu9Pxf1Alpy4pBRRd7vH+ldj35PJ/if'
    'OJmN39+RjwPSBnmg9LBIN1r9LOeAh7RMhuejez+trX//9dra91MrP/68+jXjDL0BkTloWeSl5aLY2Hzw7+XVzesba/cfbK6srZr5nYesPpi7rVnYZeBoySlz'
    'rqxfpsmrjYQmX8Gpy7JBeXK3R+MvfGwP9MUH8cCO4SaS2my8+PLu33WXc0zN0QZ04QIif+KSbtNMkvY5PqjcIrzFVVfPpt772m3ppsG22cbcqkj/SAaHeJW8'
    'YkEDhwdFAFEdBGQE6x5C99iADz7HmyPa5t2Ys1mzoqeaiWkUaB1QoyGtihljar6vImjqjkk3gyXoFkuVvzdrMCQQk07OXL0Y7a1bFusONiWTsafitKGl/YzZ'
    'kVRMfWirIL6awX5EByT56F2zmGsfWyDsAEg6Il5BA3upmSptQ7JAZEMRN345Gob2rMMoQu9lq9v4rWd5FQwQXnONmS9vrD1Y/9fyvbX1lW9XVr+6v3H925XN'
    'e+vL3yyvL6/i+ez07I2Z6enblMZGzhYaCNU2ZqPTzz+78+elz7Cx9mLlhx/X1jeJOZ+e0QXG1Szq9OhCuK5viWB9MoBUaTDlW6LRJVHdmZmmbno8vHpLlF4b'
    'eenNF5FfcjOjFY9Ls8te6WqLyPwPUEsBAhQDFAAACAgAAAAhXMCF1ZJJBwAAww4AABsAAAAAAAAAAAAAAKQBAAAAAExPRzA1X0NvbGFiX+yLpO2WieyViOuC'
    'tC5tZFBLAQIUAxQAAAAIAAAAIVyHm2IX2wkAAI4cAAASAAAAAAAAAAAAAACkAYIHAABMT0cwNV9NMDVfZm9ybXMubWRQSwECFAMUAAAACAAAACFcV4rApdQK'
    'AAA0IAAAEwAAAAAAAAAAAAAApAGNEQAATE9HMDVfTTA1X2xlc3Nvbi5tZFBLAQIUAxQAAAgIAAAAIVwFXnF7IAYAAMkMAAAkAAAAAAAAAAAAAACkAZIcAABM'
    'T0cwNV/qs7XthrVfQUnsl63tlaDsi6TsirXslYjrgrQubWRQSwECFAMUAAAICAAAACFcDXFNn/UKAACiFwAAHAAAAAAAAAAAAAAApAH0IgAATE9HMDVf6rO1'
    '7Ya1X+yekeyEseyViOuCtC5tZFBLBQYAAAAABQAFAGYBAAAjLgAAAAA='
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
_zip_path = Path.cwd() / 'LOG05_실습자료.zip'
_zip_path.write_bytes(_EMBEDDED_ZIP_BYTES)
_log05_extract_zip(_zip_path)


## 원본 가상 자료와 작성 양식

아래는 첨부된 **LOG05 실습자료 원본**에서 미션별로 추출한 자료입니다. 모의 기록에서 아직 공개되지 않은 카드나 결과는 미리 확정하지 마세요.

미션 5. 가상 보급로 투입 시나리오 기반 무인 장비 상태 점검 및 승인 실습

**가. 제공 자료 1 - 가상 보급 임무 3건의 점검 결과 [교육용가상자료]**

| 임무 | 장비·구간 | 배터리 | 링크 품질 | 적재 | 기상 | 서명 |
| --- | --- | --- | --- | --- | --- | --- |
| MA | UD-1, C 구간 | 88%(G2) | 4등급(G3) | 7단위(저울 기록) | 풍속 등급 2(G2) | 점검자·승인권자 서명 |
| MB | UV-2, B 구간 | 75%(G2) | 3등급(G3), 음영 가능 | 32단위(저울 기록) | 시정 등급 3(G3) | 점검자 서명만 |
| MC | UD-1, C 구간 | 91%(G1) | 5등급(G2) | 8단위(저울 기록) | 풍속 등급 3 예보(G4) | 점검자·승인권자 서명 |

**가. 제공 자료 2 - 경계선 사례 5종 [교육용가상자료]**

| 묶음 | 내용 | 애매한 이유 |
| --- | --- | --- |
| B1 | UD-1 배터리 출발 잔량이 합격선에서 1% 모자람 | 합격선 근처 |
| B2 | UV-2 링크 품질 합격이나 점검 로그가 1회뿐(G3) | 신뢰도 근거 약함 |
| B3 | 적재 한도 정확히 8단위, 저울 기록 없음(자기 신고) | 한도 근처이며 근거 없음 |
| B4 | 풍속 예보와 현장 측정값이 서로 다름 | 기상 정보 상충 |
| B5 | 점검표 서명은 있으나 승인권자가 부재해 대리인이 승인하겠다고 함 | 위임 범위 불분명 |

**나. 작성 서식 1 - 의사결정 트리(조건과 분기를 채운다)**

| 단계 | 확인 질문 | 예일 때 | 아니오일 때 |
| --- | --- | --- | --- |
| 1 | 4개 영역 점검이 모두 합격 기준을 충족하는가 |  |  |
| 2 | 점검 결과 신뢰도가 G3 이상이고 교차 확인이 있는가 |  |  |
| 3 | 경로 구간의 중지 조건을 조종수가 사전 확인했는가 |  |  |
| 4 | 승인권자(또는 정식 위임자)의 서명이 있는가 |  |  |
| 5 | 임무 중 사람이 중지할 수 있는 수단과 통신이 확보되었는가 |  |  |

**나. 작성 서식 2 - 분기점 조건과 책임 직책 매핑표**

| 분기점 | 조건(수치 또는 관찰 가능한 문장) | 책임 직책(가상) | 기록할 문서 |
| --- | --- | --- | --- |
| T1 |  |  |  |
| T2 |  |  |  |
| T3 |  |  |  |
| T4 |  |  |  |

**나. 작성 서식 3 - 모의 승인 기록지(임무 MA～MC와 경계선 B1～B5)**

| 대상 | 판단(승인·보류 유지·추가 확인 요청) | 근거 | 책임 직책 | 조원 합의(예·아니오) |
| --- | --- | --- | --- | --- |
| MA |  |  |  |  |
| MB |  |  |  |  |
| MC |  |  |  |  |
| B1 |  |  |  |  |
| B2 |  |  |  |  |
| B3 |  |  |  |  |
| B4 |  |  |  |  |
| B5 |  |  |  |  |

**나. 작성 서식 4 - 긴급 신속 승인 규칙: 신속 승인에서도 줄이지 않는 점검은 (   ), 사후 감사 항목은 (   ), 사후 감사 기한은 (   )이다. 중지 수단이 확보되지 않은 임무는 신속 승인 대상에서 (   ).**

**다. 점검표**

| 점검 항목 | 확인 |
| --- | --- |
| 분기점 4개의 조건을 수치 또는 관찰 가능한 문장으로 적었다 | (   ) |
| 분기점마다 책임 직책(가상)과 기록 문서를 연결했다 | (   ) |
| 임무 3건과 경계선 사례 5종 모두에 판단과 근거를 기록했다 | (   ) |
| 장비 자체 판단이나 점수만으로 승인되는 경로가 없고 사람의 승인이 필수다 | (   ) |
| 교전수칙·지휘계통 우선 문구가 있다 | (   ) |

## 📋 단계별 수행 안내와 미션 전용 AI 활용

### 활동 1: 같은 기준으로 8건을 먼저 읽는다

**무엇을 하나요?**

#### 1. 단계 개요

- **목표:**
  - 앞선 미션(M2～M4)에서 수립한 점검 항목, 신뢰도 등급(G1～G6), 책임 한계, 그리고 가상 보급 상황의 복합 지표( MA ～ MC , B1 ～ B5 )를 바탕으로, 가상 점검 및 보급 투입 대상 8건의 데이터, 수치, 서명, 위임 및 중지 수단 을 일관된 기준에 따라 면밀히 사전 검토합니다.
- **수행 방식:**
  - 각 건별 상황을 주관적으로 해석하거나 성급하게 승인 여부를 결정하지 않고, 사전에 정의된 통제 기준과 검증 수단을 대조하여 객관적으로 파악합니다.

#### 2. 세부 수행 절차

- **M2～M4 수립 기준 적용 및 8건 목록 확인:**
  - 이전 미션에서 확정된 투입 전 점검 체크리스트 기준과 신뢰도 등급, 직책별 책임 한계 규칙을 가상 보급 대상 8건의 검토에 그대로 적용합니다.
  - 수치, 서명, 위임 관계 및 중지 수단 대조: 8건의 보고서별로 계측 수치가 기준치에 부합하는지, 책임자의 적법한 서명 또는 승인 위임이 이루어졌는지, 통신 두절이나 비상 시 작동할 중지 수단이 확보되어 있는지 일일이 대조합니다.
  - 정보 공백 및 미확인 요소 사전 식별: 8건 중 데이터 누락이나 구두 보고 의존 등으로 인해 기준에 미달하는 항목이 있는지 사전에 파악하여 후속 의사결정(주판정 및 병행 요청)을 위한 준비를 마칩니다.

#### 3. 주의 및 검증 사항

- 8건을 검토할 때 단순히 목록을 훑어보는 것에 그치지 않고, 각 건마다 명시된 수치와 승인 조건이 규칙에 부합하는지 엄정하게 확인해야 합니다.
- 본 단계의 검토 결과는 다음 단계인 AI 5단계 트리·책임표 구성 및 직접 판단 과정의 기초 자료로 활용되므로 꼼꼼히 기록해야 합니다.

#### 직접 물어볼 AI 프롬프트:

- 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

#### 검토:

- 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 2: AI와 5단계 트리·책임표를 만든다

**무엇을 하나요?**

#### 1. 단계 개요

- **목표:**
  - 앞서 1단계에서 검토한 8건의 가상 보급 점검 결과를 바탕으로, AI 에이전트와 협업하여 점검 결과, G 신뢰도 근거, 조건 고지, 사람 최종 승인, 중지 수단 으로 이어지는 5단계 의사결정 트리와 직책별 책임 기록을 체계적으로 연결합니다.
- **수행 방식:**
  - 단편적이거나 주관적인 승인 판단을 지양하고, 시스템의 점검 상태부터 최종 사람의 승인 및 비상 중지 수단 확보에 이르는 전 과정을 표준화된 논리 트리와 책임 매트릭스로 구조화합니다.

#### 2. 세부 수행 절차

+ **단계별 확인 사항**
  - 5단계 의사결정 트리 설계: 1단계(점검): 배터리/유류, 통신 링크 등 투입 전 점검 항목의 수치 충족 여부 확인.
  - 2단계(G근거): 신뢰도 등급(G1～G6) 기준에 따른 데이터 신뢰성 판정.
  - 3단계(조건 고지): 비정상 상황 발생 시 임무 중지 및 복귀 조건의 사전 고지 여부 확인.
  - 4단계(사람 승인): 규정된 권한을 가진 승인권자의 적법한 최종 승인 여부 검증.
  - 5단계(중지 수단): 통신 두절이나 위협 상황 발생 시 즉시 가동할 수 있는 비상 중지·회수 수단의 확보 확인.

+ **검토 결과와 기록**
  - **직책별 책임 기록 연결:**
    - 각 단계별 의사결정 주체(군수참모, 원격 조종수, 현장 관제자 등)의 책임 범위를 매칭하여, 오류 발생 시 책임 소재를 명확히 규명할 수 있도록 책임표를 구성합니다.
    - AI 회신 내용 대조 및 보완: AI가 제안한 5단계 트리 구조와 책임 기록이 앞서 수립한 M2～M4의 규칙(최소 증거, 권한 한계 등)에 부합하는지 대조하고, 미비한 부분을 수정·보완합니다.

#### 3. 주의 및 검증 사항

- 5단계 트리 과정 중 어느 한 단계라도 누락되거나 형식적으로 넘어가서는 안 되며, 특히 4단계(사람 승인)와 5단계(중지 수단)가 명확한 근거 없이

#### 직접 물어볼 AI 프롬프트:

- 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

#### 검토:

- 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 3: 사람 역할이 직접 판단한다

**무엇을 하나요?**

#### 1. 단계 개요

- **목표:**
  - 앞선 단계에서 분석한 8건의 가상 보급 점검 결과와 AI가 제시한 분기별 설명 내용을 면밀히 대조하고, 실제 승인권자 및 관제자의 관점에서 각 건별 주판정(승인·보류·중지)과 병행 요청 사항을 직접 판정합니다.
- **수행 방식:**
  - AI의 분석 결과나 시스템의 자동 판정에만 의존하지 않고, 데이터 수치 미달이나 직책별 권한 누락 등을 우회하거나 임의로 통과시키는 사례가 없는지 엄정하게 검사하여 최종 의사결정을 내립니다.

#### 2. 세부 수행 절차

- 8건별 주판정 및 병행 요청 수립: 8건의 가상 보급 대상별로 점검 데이터의 충족 여부를 확인하여 명확한 주판정(투입 승인, 조건부 보류, 즉시 중지 등)을 내리고, 추가 검토가 필요한 사항은 병행 요청으로 기록합니다.
- **AI 분기 설명과의 심층 대조:**
  - AI 에이전트가 제시한 의사결정 분기별 논리 및 설명과 본인의 판단 결과 간의 괴리를 대조하고, 차이가 발생하는 지점의 타당성을 검토합니다.
  - 숫자 미달 및 권한 누락 우회 여부 엄정 검사: 기준 수치(배터리, 통신 링크 등)에 미달하거나 적법한 서명 및 권한 승인 체계가 누락되었음에도 이를 편법적으로 우회하여 승인 처리하려는 시도가 있는지 철저히 검증하고 바로잡습니다.

#### 3. 주의 및 검증 사항

- 현장 상황이 급박하거나 실적 압박이 있다는 이유로 수치 미달이나 권한 누락 항목을 임의로 묵인하거나 간과하지 않도록 주의해야 합니다.
- 본 단계에서 내린 최종 판단 결과와 검증 내용은 다음 단계인 긴급 절차 및 검토 상태 저장( forms/LOG05_M05_forms.md 및 sop/log05_approval_thresholds_sop.md )의 핵심 기초 자료로 활용되므로 정확하고 객관적인 기록으로 남겨야 합

#### 직접 물어볼 AI 프롬프트:

- 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

#### 검토:

- 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 4: 긴급 절차와 검토 상태를 저장한다

**무엇을 하나요?**

#### 1. 단계 개요

- **목표:**
  - 앞선 1～3단계를 통해 수행한 5단계 의사결정 트리 , 가상 점검 8건의 최종 주판정 및 병행 요청 결과 , 사람 역할의 직접 판단 내역 , 그리고 사후 감사 및 법무 검토 제안 사항 을 후속 미션인 미션 6(지휘소 관제 지침서 서식 개발) , 미션 9(피드백 반영 체크리스트 보완) , 미션 10(표준 체크리스트 완성)으로 정확하고 누락 없이 인계합니다.
- **수행 방식:**
  - 개별 미션의 성과물로 단절되지 않고, 도출된 긴급 중지 절차와 검토 상태를 후속 실습 및 최종 지침서 편성에 온전히 반영할 수 있도록 정돈된 형태로 저장 및 이관합니다.

#### 2. 세부 수행 절차

- 5단계 의사결정 결과 및 8건의 판정 이력 정리: 점검 결과, G 신뢰도 근거, 조건 고지, 사람 승인, 중지 수단 등 5단계 트리에 따라 8건의 가상 보급 대상별로 확정된 주판정(승인·보류·중지)과 검토 사유를 최종 문서 형태로 정리합니다.
- 사후 감사 및 법무 검토 제안 사항 도출: 판단 과정에서 규정 해석의 모호함이나 예외 조치로 인해 향후 감사나 법무 검토가 필요하다고 판단되는 쟁점들을 정리하여 제안 사항으로 명시합니다.
- **필수 파일 경로 연계 및 저장:**
  - 정리된 내용을 sop/log05_approval_thresholds_sop.md , LOG05_M05_forms.md , forms/LOG05_M05_forms.md 등 지정된 결과 파일 경로에 빠짐없이 반영하고 저장합니다.
  - 후속 미션(M6, M9, M10) 연계 지점 확정: 본 미션에서 확정된 긴급 절차와 검토 상태가 미션 6의 실시간 관제 및 긴급 중지 지침서, 미션 9의 체크리스트 보완, 그리고 미션 10의 최종 표준 체크리스트 완성 과정에 직접적인 기준 자료로

#### 직접 물어볼 AI 프롬프트:

- 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

#### 검토:

- 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

## 제출 전에 확인할 기준

**평가 항목 1:** [정량적 평가] [필수] 분기점·책임 매핑·모의 승인 — 승인/보류 조건 분기점(Threshold)을 100% 명시하고 책임자 직책 매핑을 완료했는가, 경계선 사례 5종과 가상 임무 3건 모두에 판단과 근거를 기록했는가, 교전수칙·지휘계통 우선 원칙과 사람의 승인을 지키고 긴급 신속 승인과 사후 감사 규정을 보완했는가.

- PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다.
- 목표미달·0기준·미측정은그대로표시합니다.
- 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

**평가 항목 2:** [정성적 평가] Why/How/What: 본인 계획·실제 대화·근거 검토3건 이상·보완·재확인·최종 반영.

- PASS 증거 가상 보급로 투입 시나리오에서 무인 장비의 점검 결과를 보고 투입 승인·보류·중지를 판단하는 분기점과 책임 직책을 확정하고 실습한다.의판단이제공자료·역할책임·교육용가정과연결됩니다.
- 사실/제안/미확인과실제/모의수행을구분합니다.
- 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

**평가 항목 3:** [진행 추적 및 완료 기준] 실제 조원·두 평가자·2분 판단·20분 문서점검·3분 발표·실제 교관 확인을 AI 모의와 구분한다.

- 미실시를 PASS로 쓰지 않는다.
- 파일: sop/log05_approval_thresholds_sop.md, LOG05_M05_forms.md, forms/LOG05_M05_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
- 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
- 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**중요:** 정답·실제 측정값·사람의 서명이나 승인을 AI가 만들어 주지 않도록 합니다.

In [ ]:
MISSION_JSON = '{"code": "LOG05_m05", "steps": [{"title": "같은 기준으로 8건을 먼저 읽는다"}, {"title": "AI와 5단계 트리·책임표를 만든다"}, {"title": "사람 역할이 직접 판단한다"}, {"title": "긴급 절차와 검토 상태를 저장한다"}]}'
# LOG05 학습 기록 도구 (표준 Python/Jupyter 환경용)
import json, zipfile, io, os, datetime
from pathlib import Path
from IPython.display import display, Markdown
def _load_notebook_widgets():
    from importlib import import_module, invalidate_caches
    import subprocess
    import sys

    try:
        return import_module('ipywidgets')
    except ImportError:
        print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

    try:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
             '--retries', '0', '--timeout', '10', 'ipywidgets'],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
        )
        invalidate_caches()
        return import_module('ipywidgets')
    except (subprocess.SubprocessError, OSError, ImportError):
        print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
        print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
        print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
        return None

widgets = _load_notebook_widgets()
if widgets is not None:
    MISSION_DATA = json.loads(MISSION_JSON)
    CODE = MISSION_DATA['code']
    OUT_DIR = Path(CODE + '_results')
    OUT_DIR.mkdir(exist_ok=True)
    BACKUP = OUT_DIR / (CODE + '_record.json')
    fields = {}
    for no, item in enumerate(MISSION_DATA['steps'], 1):
        fields['activity_%02d' % no] = widgets.Textarea(
            value='', placeholder='내 판단, 확인한 자료 번호, AI 제안과 수정 이유를 직접 작성하세요.',
            description='활동 %s' % no, layout=widgets.Layout(width='98%',height='130px'),
            style={'description_width':'75px'})
    for key, label in [('first_thought','AI에게 묻기 전 내 판단'), ('ai_request','실제로 보낸 AI 요청 전체'),
                       ('ai_response','실제로 받은 AI 답변 전체'), ('source_check','원자료 대조 3건과 수정 이유'),
                       ('final_result','최종 작성 결과와 근거'), ('limits','미실시/미측정/확인 필요 및 교관 검토 상태')]:
        fields[key] = widgets.Textarea(value='', description=label,
           layout=widgets.Layout(width='98%',height='125px'),style={'description_width':'190px'})
    status = widgets.HTML(value='<b>아직 저장하지 않았습니다.</b> 작성과 교관 승인은 별개입니다.')
    restore_input = widgets.Textarea(description='복원 JSON',placeholder='백업 JSON의 내용을 붙여넣으세요.',layout=widgets.Layout(width='98%',height='100px'),style={'description_width':'110px'})

    def payload():
        return {'schema':'log05-student-v1','mission':CODE,'saved_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
                'answers':{key:w.value for key,w in fields.items()},
                'status':'작성 기록 (학습 PASS 또는 교관 승인 아님)'}

    def save_record(_=None):
        obj=payload()
        if BACKUP.exists():
            stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
            BACKUP.rename(OUT_DIR / (CODE+'_previous_'+stamp+'.json'))
        BACKUP.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8')
        status.value='<b>로컬 저장 완료:</b> '+str(BACKUP)+' (PASS 아님)'

    def export_json(_=None):
        save_record()
        raw=BACKUP.read_bytes()
        try:
            from google.colab import files
            files.download(str(BACKUP))
        except ImportError:
            print('Colab이 아닙니다. 저장 위치:',BACKUP.resolve())

    def restore_record(_=None):
        try:
            obj=json.loads(restore_input.value)
            if obj.get('mission')!=CODE or not isinstance(obj.get('answers'),dict):raise ValueError('다른 미션 또는 형식이 맞지 않는 백업입니다.')
            save_record()  # 복원 전 백업 저장
            for key,w in fields.items():w.value=str(obj['answers'].get(key,''))
            status.value='<b>복원 적용:</b> 기존 작성 내용은 이전 JSON으로 보관했습니다. 다시 저장하세요.'
        except Exception as exc:status.value='<b>복원 오류:</b> '+str(exc)

    def export_review_zip(_=None):
        save_record()
        target=OUT_DIR / (CODE+'_review.zip')
        with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as zz:
            zz.write(BACKUP,arcname=BACKUP.name)
            text='\n\n'.join('## '+key+'\n'+val for key,val in payload()['answers'].items())
            zz.writestr(CODE+'_work.md',text)
            zz.writestr('README.txt','검토용 보관 파일. 실제 평가 PASS 또는 교관 승인 자료가 아닙니다.')
        try:
            from google.colab import files
            files.download(str(target))
        except ImportError:
            print('Colab이 아닙니다. ZIP 위치:',target.resolve())

    for i,item in enumerate(MISSION_DATA['steps'],1):
        display(Markdown('### 활동 %d. %s' % (i,item['title'])))
        display(fields['activity_%02d' % i])
    for key,label in [('first_thought','최초 판단'),('ai_request','AI 요청 원문'),('ai_response','AI 답변 원문'),
     ('source_check','원자료 대조'),('final_result','최종 결과'),('limits','한계 및 실제 검토 상태')]:
        display(Markdown('### '+label));display(fields[key])
    buttons=[]
    for title,handler in [('작성 내용 저장',save_record),('작성 JSON 내려받기',export_json),('검토용 ZIP 내려받기',export_review_zip),('작성 JSON 복원',restore_record)]:
        b=widgets.Button(description=title,button_style='info');b.on_click(handler);buttons.append(b)
    display(widgets.HBox(buttons[:2]));display(widgets.HBox(buttons[2:]));display(restore_input);display(status)


## 마무리

- 작성 JSON과 검토용 ZIP을 내려받아 보관하세요.
- ZIP은 **검토용**이며 평가 PASS나 교관 승인 증명이 아닙니다.
- 전 단계 결과가 필요하면 본인의 실제 이전 기록을 사용하고, 기록이 없으면 **미실시**로 남기세요.